# LedgerLab Minimal HF TRL Training Demo

This notebook demonstrates the minimal training path for the LedgerLab OpenEnv environment using **Hugging Face TRL**.

What it uses:
- Environment: `ReactAgentEnv` / LedgerLab
- Trainer: `trl.GRPOTrainer`
- Entrypoint: `training/train_finbench_grpo.py`
- Base model: `Qwen/Qwen3-1.7B`

This is the same script family we used for the real H100 runs. The Colab notebook is a minimal reproducible wrapper around that script for hackathon submission.


## Notes

- For the hackathon requirement, this satisfies the **HF TRL** training-script requirement.
- Colab is used here as a reproducible demo notebook, not as the primary long-run training environment.
- The real larger runs were executed on an H100 because this environment is long-horizon and tool-using.
- If this notebook is slow on free Colab, reduce task count further or switch to Colab Pro / A100.


In [ ]:
!rm -rf /content/ledgerlab-openenv
!git clone https://github.com/vivek100/ledgerlab-openenv /content/ledgerlab-openenv
%cd /content/ledgerlab-openenv/ReactAgentEnv


In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install --index-url https://download.pytorch.org/whl/cu124 torch
!python -m pip install -r training/requirements-smoke.txt
!python -m ipykernel install --sys-prefix --name python3


In [ ]:
import os
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['WANDB_PROJECT'] = os.environ.get('WANDB_PROJECT', 'ledgerlab')
# Optional: uncomment if you want Colab logging
# os.environ['WANDB_API_KEY'] = 'YOUR_WANDB_API_KEY'
# os.environ['WANDB_ENTITY'] = 'YOUR_WANDB_ENTITY'

Path('traces').mkdir(exist_ok=True)
Path('data/_persistent_memory').mkdir(parents=True, exist_ok=True)
print('Environment prepared.')


## Minimal HF TRL Run

This calls the real LedgerLab training script with the conservative settings that actually worked for us:
- `max_train_tasks=4`
- `repeats_per_task=1`
- `num_train_epochs=1`
- `max_turns=6`
- `num_generations=2`
- `max_completion_length=96`
- `max_prompt_length=1536`
- `--no-vllm`


In [ ]:
!python training/train_finbench_grpo.py \
  --model-name Qwen/Qwen3-1.7B \
  --max-train-tasks 4 \
  --repeats-per-task 1 \
  --num-train-epochs 1 \
  --max-turns 6 \
  --num-generations 2 \
  --max-completion-length 96 \
  --max-prompt-length 1536 \
  --save-steps 5 \
  --output-dir /content/ledgerlab_outputs/model \
  --no-vllm


In [ ]:
!find /content/ledgerlab_outputs -maxdepth 3 -type f | sort | head -n 100


## What This Proves

- The environment can be instantiated from the repo.
- LedgerLab training uses **HF TRL** (`GRPOTrainer`).
- The same actual project script can be run from a notebook.

For larger or repeated runs, use the Northflank H100 jobs and the W&B-tracked training flow from the repo.
